In [33]:
# Importation des modules
# Import bibliothèque de manipulation de dataframe
import pandas as pd

# Import des bibliothèques de viz
import matplotlib.pyplot as plt
import seaborn as sns

# Import split data
from sklearn.model_selection import train_test_split

# Import modèles de ML Supervisé Régression
from sklearn.linear_model import LinearRegression

# Import modèles de ML Supervisé Classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Import modèle de ML NON Supervisé
from sklearn.neighbors import NearestNeighbors

# Import des métriques
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Import outil standardisation de la donnée
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

# Import pipeline
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

# Gestion des warnings
import warnings

In [34]:
# Custom transformer for MultiLabelBinarizer
class MultiLabelBinarizerPipelineFriendly(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

In [32]:
# Récuperation du df
df = pd.read_csv('../ressources/df_v2.csv', sep=';', encoding='utf-8')
df

,frenchTitle,genres,averageRating,numVotes,decade
0,J'accuse,"Horror, War, Romance, Drama, History",7.7,2240,1910
1,Coeur fidèle,"Romance, Drama",7.4,1540,1920
2,L'inhumaine,"Romance, ScienceFiction, Drama, Mystery",7.2,1134,1920
3,Les cheveux d'or,"Thriller, Drama, Crime, Mystery",7.3,14289,1920
4,Nana,"Romance, Drama",6.6,1011,1920
...,...,...,...,...,...
12660,Herself,Drama,7.0,5099,2020
12661,Enemy Lines,"Drama, Action, War",4.6,2041,2020
12662,Le lion,Comedy,5.5,1497,2020
12663,Safeguard,"Thriller, Adventure, Action, Crime",3.6,263,2020


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Preprocessor
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [36]:
X = df.drop(columns=['frenchTitle'])
films_non_standardise = X.iloc[:3]

In [39]:
# Preprocessor pour standardiser les colonnes numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('encoder', OrdinalEncoder(), ['decade']),  # Remplacez par vos colonnes numériques
        ('genres', MultiLabelBinarizerPipelineFriendly(), 'genres'),
    ],
    remainder='passthrough'
)


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Pipeline
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [40]:
# Création du pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('knn', NearestNeighbors(n_neighbors=5))
    ]
)

pipeline.fit(X)

c:\Users\User\Documents\Projet2\WildCodeSchool-Projet2\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder', OrdinalEncoder(),
                                                  ['decade']),
                                                 ('genres',
                                                  MultiLabelBinarizerPipelineFriendly(),
                                                  'genres')])),
                ('knn', NearestNeighbors())])

In [ ]:

point_test = X.iloc[:3]
# Prédiction des voisins les plus proches
distances, indices = pipeline.named_steps['knn'].kneighbors(point_test)
# Affichage des indices des voisins les plus proches
print("Indices des voisins les plus proches :", indices)
# Affichage des distances des voisins les plus proches
print("Distances des voisins les plus proches :", distances)


Indices des voisins les plus proches : [[10937 12550  6192 12514 11569]
 [10937 12550  6192 12514 11569]
 [10937 12550  6192 12514  9646]]
Distances des voisins les plus proches : [[2936.41328494 2936.91489492 2936.93849101 2937.01591756 2937.04969825]
 [2455.28084952 2455.70917863 2455.73752058 2455.77298829 2455.89889857]
 [2225.00335505 2225.3660036  2225.39781837 2225.40037971 2225.58129261]]


c:\Users\User\Documents\Projet2\WildCodeSchool-Projet2\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but NearestNeighbors was fitted without feature names
  warnings.warn(


In [ ]:
print(f"\nRecherche des 5 plus proches voisins pour 3 films (basé sur les features standardisées de X_class):")
for i in range(len(point_test)):
    # Récupérer l'index original du point exemple dans le DataFrame df
    original_index = films_non_standardise.index[i]
    print(f"\n--- Voisinage pour Film Exemple {i+1} (Index original: {original_index}) ---")
    print(f"  Note moyenne: {df.loc[original_index, 'averageRating']}")

    # Les 'indices' renvoyés par kneighbors sont les positions (0, 1, 2...) dans X_knn_scaled
    print(f"  Indices des voisins dans X_knn_scaled: {indices[i]}")
    # Les distances correspondantes
    print(f"  Distances euclidiennes aux voisins: {distances[i]}")

    # Pour afficher les informations des voisins, il faut retrouver leurs index originaux dans df.
    # Comme X_knn_scaled a été créé à partir de X_class dans le même ordre, on peut utiliser
    # les indices[i] pour sélectionner les lignes correspondantes dans X_class, puis récupérer leurs index originaux.
    neighbor_original_indices = X.iloc[indices[i]].index
    print(f"  Index originaux des voisins dans le DataFrame: {list(neighbor_original_indices)}")

    # Afficher quelques caractéristiques clés des voisins trouvés à partir du df original
    print("  Infos sur les voisins trouvés (depuis df original):")
    # Sélectionner les lignes des voisins dans df et quelques colonnes pertinentes
    neighbor_info = df.loc[neighbor_original_indices][['averageRating', 'numVotes', 'decade']]
    print(neighbor_info)

In [42]:
titres = ['Inception', 'Titanic', 'Avatar']  # ou d’autres

for titre in titres:
    film_cible = df[df['frenchTitle'].str.contains(titre, case=False, na=False)]
    if film_cible.empty:
        print(f"Film '{titre}' non trouvé.")
        continue

    idx_film = film_cible.index[0]
    film_non_standardise = df.drop(columns=['frenchTitle']).loc[[idx_film]]
    film_transforme = pipeline.named_steps['preprocessor'].transform(film_non_standardise)
    distances, indices = pipeline.named_steps['knn'].kneighbors(film_transforme)

    print(f"\n🎬 Film : {df.loc[idx_film, 'frenchTitle']} (Index: {idx_film})")
    print(f"  Note moyenne : {df.loc[idx_film, 'averageRating']}")
    neighbor_original_indices = X.iloc[indices[0]].index
    neighbor_info = df.loc[neighbor_original_indices][['frenchTitle', 'averageRating', 'numVotes', 'decade', 'genres']]
    print("  Voisins :")
    display(neighbor_info)


🎬 Film : Inception (Index: 7143)
  Note moyenne : 8.8
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres
7143,Inception,8.8,2682513,2010,"Adventure, ScienceFiction, Action, Sci-Fi"
5207,The Dark Knight : Le Chevalier noir,9.0,3019825,2000,"Thriller, Drama, Action, Crime"
5530,Interstellar,8.7,2343939,2010,"Adventure, ScienceFiction, Drama, Sci-Fi"
3265,Gladiator,8.5,1759137,2000,"Adventure, Drama, Action"
4472,Inglourious Basterds,8.4,1682180,2000,"Thriller, Adventure, Drama, War"



🎬 Film : Titanic Town (Index: 3034)
  Note moyenne : 6.4
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres
3034,Titanic Town,6.4,466,1990,Drama
9926,My Feral Heart,6.9,466,2010,Drama
10842,Premiers crus,5.8,465,2010,Drama
3878,Betelnut Beauty,6.4,466,2000,"Romance, Drama"
7197,Orly,6.0,468,2010,Drama



🎬 Film : Avatar (Index: 5390)
  Note moyenne : 7.9
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres
5390,Avatar,7.9,1430677,2000,"Fantasy, Adventure, ScienceFiction, Action"
5294,Le Prestige,8.5,1514342,2000,"ScienceFiction, Drama, Mystery, Sci-Fi"
2376,Léon,8.5,1302523,1990,"Drama, Action, Crime"
1988,Terminator 2 : Le Jugement dernier,8.6,1232359,1990,"Thriller, ScienceFiction, Adventure, Action, S..."
4553,Batman Begins,8.2,1646214,2000,"Drama, Action, Crime"


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Entrainement du modele
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------